# Binary Classification with a Bank Dataset
### Playground Series - Season 5, Episode 8

### Libraries Import

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from tqdm import tqdm
from itertools import combinations
import gc

import xgboost as xgb

from sklearn.manifold import TSNE as sklearn_TSNE

import optuna
import torch
import copy
import itertools
import warnings

warnings.filterwarnings('ignore')

In [4]:
import os
for dirname, _, filenames in os.walk('./input/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./input/test.csv
./input/train.csv
./input/archive\bank-full.csv


### Data Import & Info

In [6]:
train = pd.read_csv("./input/train.csv", index_col='id')
test = pd.read_csv("./input/test.csv", index_col='id')

In [7]:
TARGET = 'y'
NUMS = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
CATS = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

In [8]:
# Add match_p feature from external dataset
orig = pd.read_csv('./input/archive/bank-full.csv', delimiter=';')

In [9]:
orig['y'] = orig['y'].replace({'yes': 1, 'no': 0})

train[CATS] = train[CATS].astype('category')
test[CATS] = test[CATS].astype('category')
orig[CATS] = orig[CATS].astype('category')

TE_columns = []

columns = NUMS + CATS

for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        orig[name] = orig[cols[0]].astype(str)
        for col in cols[1:]:
            orig[name] = orig[name] + '_' + orig[col].astype(str)
        
        combined = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        orig[name] = combined[len(train) + len(test):]

        TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

100%|██████████| 120/120 [02:11<00:00,  1.09s/it]


In [10]:
def target_encode(train, valid, test, col, target=TARGET, kfold=5, smooth=20, agg='mean'):
    train['kfold'] = ((train.index) % kfold)
    col_name = '_'.join(col)
    train[f'TE_{agg.upper()}_' + col_name] = 0.
    for i in range(kfold):
        df_tmp = train[train['kfold'] != i]
        if agg == 'mean': mn = train[target].mean()
        elif agg == 'median': mn = train[target].median()
        elif agg == 'min': mn = train[target].min()
        elif agg == 'max': mn = train[target].max()
        elif agg == 'nunique': mn = 0
        df_tmp = df_tmp[col + [target]].groupby(col).agg([agg, 'count']).reset_index()
        df_tmp.columns = col + [agg, 'count']
        if agg == 'nunique':
            df_tmp['TE_tmp'] = df_tmp[agg] / df_tmp['count']
        else:
            df_tmp['TE_tmp'] = ((df_tmp[agg] * df_tmp['count']) + (mn * smooth)) / (df_tmp['count'] + smooth)
        df_tmp_m = train[col + ['kfold', f'TE_{agg.upper()}_' + col_name]].merge(df_tmp, how='left', left_on=col, right_on=col)
        df_tmp_m.loc[df_tmp_m['kfold'] == i, f'TE_{agg.upper()}_' + col_name] = df_tmp_m.loc[df_tmp_m['kfold'] == i, 'TE_tmp']
        train[f'TE_{agg.upper()}_' + col_name] = df_tmp_m[f'TE_{agg.upper()}_' + col_name].fillna(mn).values

    df_tmp = train[col + [target]].groupby(col).agg([agg, 'count']).reset_index()
    if agg == 'mean': mn = train[target].mean()
    elif agg == 'median': mn = train[target].median()
    elif agg == 'min': mn = train[target].min()
    elif agg == 'max': mn = train[target].max()
    elif agg == 'nunique': mn = 0
    df_tmp.columns = col + [agg, 'count']
    if agg == 'nunique':
        df_tmp['TE_tmp'] = df_tmp[agg] / df_tmp['count']
    else:
        df_tmp['TE_tmp'] = ((df_tmp[agg] * df_tmp['count']) + (mn * smooth)) / (df_tmp['count'] + smooth)
    df_tmp_m = valid[col].merge(df_tmp, how='left', left_on=col, right_on=col)
    valid[f'TE_{agg.upper()}_' + col_name] = df_tmp_m['TE_tmp'].fillna(mn).values
    valid[f'TE_{agg.upper()}_' + col_name] = valid[f'TE_{agg.upper()}_' + col_name].astype('float32')

    df_tmp_m = test[col].merge(df_tmp, how='left', left_on=col, right_on=col)
    test[f'TE_{agg.upper()}_' + col_name] = df_tmp_m['TE_tmp'].fillna(mn).values
    test[f'TE_{agg.upper()}_' + col_name] = test[f'TE_{agg.upper()}_' + col_name].astype('float32')

    train = train.drop('kfold', axis=1)
    train[f'TE_{agg.upper()}_' + col_name] = train[f'TE_{agg.upper()}_' + col_name].astype('float32')

    return (train, valid, test)

def count_encode(train, valid, test, col):
    counts = train[col].value_counts()

    train[f'CE_{col}'] = train[col].map(counts)
    valid[f'CE_{col}'] = valid[col].map(counts).fillna(0)
    test[f'CE_{col}'] = test[col].map(counts).fillna(0)
    return (train, valid, test)

In [11]:
oof = np.zeros(len(train))
pred = np.zeros(len(test))

# ---- Configuration ----
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

def objective(trial):
    auc_scores = []

    # Hyperparamètres à optimiser
    param_grid = {
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "subsample": trial.suggest_float("subsample", 0.3, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10.0, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
    }

    for idx, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET])):
        X_train, X_val = train.loc[train_idx, FEATURES], train.loc[val_idx, FEATURES]
        y_train, y_val = train.loc[train_idx, TARGET], train.loc[val_idx, TARGET]
        X_test = test.copy()

        # Ajout des données "orig"
        X_train = pd.concat([X_train, orig[FEATURES]])
        y_train = pd.concat([y_train, orig[TARGET]])

        # Encodages
        for col in TE_columns:
            X_train, X_val, X_test = target_encode(
                pd.concat([X_train, y_train], axis=1), 
                X_val, X_test, [col], smooth=10, agg='mean'
            )
            X_train = X_train.drop(TARGET, axis=1)
            X_train, X_val, X_test = count_encode(X_train, X_val, X_test, col)
            X_train = X_train.drop(col, axis=1)
            X_val = X_val.drop(col, axis=1)
            X_test = X_test.drop(col, axis=1)

        model = XGBClassifier(
            **param_grid,
            n_estimators=10000,
            objective="binary:logistic",
            eval_metric="auc",
            learning_rate=0.01,
            early_stopping_rounds=1000,
            random_state=42,
            enable_categorical=True,
            device="cuda",
            n_jobs=-1
        )

        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        y_pred = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, y_pred))

        del model, X_train, X_val, y_train, y_val, X_test
        gc.collect()

    return sum(auc_scores) / len(auc_scores)


# ---- Lancer l'optimisation Optuna ----
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)  # tu peux augmenter le nombre de trials

print("Meilleurs paramètres :", study.best_params)
print("Meilleur score AUC moyen :", study.best_value)

[I 2025-08-16 11:12:43,144] A new study created in memory with name: no-name-f7fba94a-de3c-4ecc-83d5-bd84f9b4b7a6
[W 2025-08-16 13:46:27,211] Trial 0 failed with parameters: {'colsample_bytree': 0.3910673730919758, 'subsample': 0.3640453451610804, 'reg_lambda': 2.9374614389372833, 'reg_alpha': 0.13995268692809026, 'max_depth': 5} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\gabri\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\gabri\AppData\Local\Temp\ipykernel_39248\2821856956.py", line 53, in objective
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
  File "C:\Users\gabri\anaconda3\Lib\site-packages\xgboost\core.py", line 729, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "C:\Users\gabri\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1682, in fit
    sel

KeyboardInterrupt: 

In [17]:
best_params = {'colsample_bytree': 0.3910673730919758, 'subsample': 0.3640453451610804, 'reg_lambda': 2.9374614389372833, 'reg_alpha': 0.13995268692809026, 'max_depth': 5}

In [ ]:
# ---- Entraînement final avec les meilleurs paramètres ----
oof = np.zeros(len(train))
pred = np.zeros(len(test))

for idx, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET])):
    X_train, X_val = train.loc[train_idx, FEATURES], train.loc[val_idx, FEATURES]
    y_train, y_val = train.loc[train_idx, TARGET], train.loc[val_idx, TARGET]
    X_test = test.copy()

    X_train = pd.concat([X_train, orig[FEATURES]])
    y_train = pd.concat([y_train, orig[TARGET]])

    for col in TE_columns:
        X_train, X_val, X_test = target_encode(
            pd.concat([X_train, y_train], axis=1), 
            X_val, X_test, [col], smooth=10, agg='mean'
        )
        X_train = X_train.drop(TARGET, axis=1)
        X_train, X_val, X_test = count_encode(X_train, X_val, X_test, col)
        X_train = X_train.drop(col, axis=1)
        X_val = X_val.drop(col, axis=1)
        X_test = X_test.drop(col, axis=1)

    model = XGBClassifier(
        **best_params,
        n_estimators=10000,
        objective="binary:logistic",
        eval_metric="auc",
        learning_rate=0.01,
        early_stopping_rounds=200,
        random_state=42,
        enable_categorical=True,
        device="cuda",
        n_jobs=-1
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test)[:, 1]

    print(f"Fold {idx + 1}: {roc_auc_score(y_val, oof[val_idx])}")

    del model, X_train, X_val, y_train, y_val, X_test
    gc.collect()

pred /= 5
print(f"CV AUC: {roc_auc_score(train[TARGET], oof)}")

In [ ]:
submission = pd.read_csv('./output/sample_submission.csv')
submission['y'] = pred
submission.to_csv('to_submit.csv', index=False)

In [ ]:
submission